<a href="https://colab.research.google.com/github/JEWONMOON/NZFC_Paper/blob/main/QWEN2_5_7B_NZFC_VS_TQ%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NZFC Fix C vs TurboQuant — Qwen2.5-7B

## 모델 사양
- `Qwen/Qwen2.5-7B` — 표준 GQA transformer (Qwen3.5와 달리 hybrid 아님)
- `H_KV=8, head_dim=128, layers=28`
- KV tensor per layer: `[1, 8, T, 128]`

## 비교 대상
| 방법 | 설명 |
|---|---|
| **TQ-MSE** | TurboQuant MSE-only — QJL 없음, 공정한 강한 기준선 |
| **TQ-Prod** | 논문의 TurboQuant — MSE + 1-bit QJL residual |
| **NZFC-Orig** | 전체 재압축 기준 |
| **NZFC-FixC** | scatter_add_ + K-only cdist + pre-alloc 버퍼 |

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers>=4.45.0', 'datasets', 'accelerate', 'tabulate'])

import pathlib, time, math, json, cProfile, pstats, io, gc, warnings
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE   = torch.float32
OUT_DIR = pathlib.Path('/kaggle/working/fixc_qwen25_7b')
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.set_grad_enabled(False)

print(f'torch={torch.__version__}  device={DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  {p.total_memory/1e9:.1f} GB')

torch=2.10.0+cu128  device=cuda
GPU: Tesla T4  15.6 GB


## Section 1 — Qwen2.5-7B 로드 & KV 캡처

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
from datasets import load_dataset

MODEL_NAME   = 'Qwen/Qwen2.5-7B'
MAX_TOKENS   = 1024
CHUNK_SIZE   = 8
BENCH_LAYERS = [6, 13, 20, 27]

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# device_map='auto': T4 16GB에 맞게 자동 분배 (가중치 ~14.4GB)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True)
model.eval()

cfg = model.config
N_KV_HEADS = cfg.num_key_value_heads   # 8
N_Q_HEADS  = cfg.num_attention_heads   # 28
HEAD_DIM   = cfg.hidden_size // N_Q_HEADS  # 128
N_LAYERS   = cfg.num_hidden_layers     # 28

print(f'kv_heads={N_KV_HEADS}  q_heads={N_Q_HEADS}  head_dim={HEAD_DIM}  layers={N_LAYERS}')
if DEVICE.type == 'cuda':
    print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── wikitext-103 토큰화 ──────────────────────────────────────────────────────
print('Loading wikitext-103...')
ds = load_dataset('wikitext', 'wikitext-103-v1', split='validation')
ids = []
for r in ds:
    if isinstance(r['text'], str) and r['text'].strip():
        ids.extend(tokenizer.encode(r['text'], add_special_tokens=False))
    if len(ids) >= MAX_TOKENS: break
ids = ids[:MAX_TOKENS]
input_ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
print(f'{len(ids)} tokens')

# ── DynamicCache.update monkey-patch ─────────────────────────────────────────
# Qwen2.5는 표준 DynamicCache를 사용 (Qwen3.5-9B의 HybridCache와 다름)
_kv_buf: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}
_orig_update = DynamicCache.update

def _capture_hook(self, key_states, value_states, layer_idx, cache_kwargs=None):
    li = int(layer_idx)
    if li in BENCH_LAYERS:
        _kv_buf[li] = (
            key_states.detach().cpu().float(),
            value_states.detach().cpu().float()
        )
    return _orig_update(self, key_states, value_states, layer_idx, cache_kwargs)

DynamicCache.update = _capture_hook
print(f'DynamicCache.update patched ✓')

# ── KV 캡처 루프 ─────────────────────────────────────────────────────────────
kv_dump = {li: {'k_chunks': [], 'v_chunks': []} for li in BENCH_LAYERS}
past_kv = None
n_chunks = math.ceil(MAX_TOKENS / CHUNK_SIZE)
print(f'Capturing KV ({n_chunks} chunks)...')

for i in range(0, input_ids.shape[1], CHUNK_SIZE):
    chunk = input_ids[:, i:i+CHUNK_SIZE]
    _kv_buf.clear()
    with torch.no_grad():
        out = model(input_ids=chunk, use_cache=True, past_key_values=past_kv)
    past_kv = out.past_key_values
    for li in BENCH_LAYERS:
        if li in _kv_buf:
            k_new, v_new = _kv_buf[li]
            kv_dump[li]['k_chunks'].append(k_new)
            kv_dump[li]['v_chunks'].append(v_new)
    if (i // CHUNK_SIZE) % 32 == 0:
        print(f'  {i//CHUNK_SIZE}/{n_chunks}')

DynamicCache.update = _orig_update
print('DynamicCache.update restored ✓')

del past_kv, out; gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

print('\nKV dump:')
for li in BENCH_LAYERS:
    chunks = kv_dump[li]['k_chunks']
    if not chunks:
        print(f'  layer {li:2d}: EMPTY')
    else:
        total_t = sum(c.shape[2] for c in chunks)
        print(f'  layer {li:2d}: {total_t} tokens  shape={tuple(chunks[0].shape)}')
        # shape 검증
        assert chunks[0].shape[1] == N_KV_HEADS, \
            f'H_KV mismatch: {chunks[0].shape[1]} != {N_KV_HEADS}'
        assert chunks[0].shape[3] == HEAD_DIM, \
            f'HEAD_DIM mismatch: {chunks[0].shape[3]} != {HEAD_DIM}'
print('Shape validation ✓')

Loading Qwen/Qwen2.5-7B...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

wikitext-103-v1/validation-00000-of-0000(…):   0%|          | 0.00/655k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

1024 tokens
DynamicCache.update patched ✓
Capturing KV (128 chunks)...
  0/128
  32/128
  64/128
  96/128
DynamicCache.update restored ✓

KV dump:
  layer  6: 1024 tokens  shape=(1, 4, 8, 128)
  layer 13: 1024 tokens  shape=(1, 4, 8, 128)
  layer 20: 1024 tokens  shape=(1, 4, 8, 128)
  layer 27: 1024 tokens  shape=(1, 4, 8, 128)
Shape validation ✓


## Section 2 — TurboQuant (TQMse + TQProd)

In [ ]:
_CB: Dict[Tuple[int,int], np.ndarray] = {}

def lloyd_max_cb(bits, gs=100_001, gl=8.0, mi=200, tol=1e-8):
    if bits <= 0: return np.zeros((0,), dtype=np.float64)
    key = (bits, gs)
    if key in _CB: return _CB[key].copy()
    K = 2**bits; g = np.linspace(-gl, gl, gs, dtype=np.float64)
    pdf = np.exp(-0.5*g*g) / math.sqrt(2*math.pi)
    c = np.interp((np.arange(K)+0.5)/K, np.cumsum(pdf)/np.cumsum(pdf)[-1], g)
    for _ in range(mi):
        bnd = np.empty(K+1); bnd[0]=-np.inf; bnd[-1]=np.inf
        bnd[1:-1] = 0.5*(c[:-1]+c[1:]); nc = c.copy()
        for k in range(K):
            m=(g>=bnd[k])&(g<bnd[k+1]); w=pdf[m].sum()
            if w>0: nc[k]=(g[m]*pdf[m]).sum()/w
        if np.max(np.abs(nc-c))<tol: c=nc; break
        c=nc
    _CB[key]=c.copy(); return c

@dataclass
class TQEnc:
    indices: Optional[torch.Tensor]; qjl_signs: Optional[torch.Tensor]
    norms: torch.Tensor; residual_norms: Optional[torch.Tensor]
    dequantized: Optional[torch.Tensor]; storage_bytes: float
    mode: str; bits_per_dim: float

class TQBase:
    def __init__(self, dim, *, bits, seed=0, dtype=torch.float32):
        self.dim=int(dim); self.bits=int(bits); self.dtype=dtype
        gen=torch.Generator(); gen.manual_seed(int(seed))
        G=torch.randn(dim,dim,generator=gen,dtype=torch.float64)
        Q,R=torch.linalg.qr(G); sign=torch.sign(torch.diag(R)); sign[sign==0]=1.0
        self.W=(Q*sign).to(dtype=dtype); self.sigma=1.0/math.sqrt(dim)
    def _nr(self, x):
        n=torch.clamp(torch.linalg.norm(x.float(),dim=-1,keepdim=True),min=1e-12)
        return (x.float()/n).to(self.dtype), n.squeeze(-1).to(self.dtype)

class TQMse(TQBase):
    """TurboQuant MSE-only (QJL 없음) — 공정한 강한 기준선"""
    def __init__(self, dim, *, bits, seed=0, dtype=torch.float32):
        super().__init__(dim, bits=bits, seed=seed, dtype=dtype)
        self.centroids = torch.tensor(
            (lloyd_max_cb(bits) if bits>0 else np.zeros((0,),dtype=np.float64))*self.sigma,
            dtype=dtype)
    @torch.no_grad()
    def encode(self, x, ret_deq=False):
        u,n=self._nr(x)
        if self.bits==0:
            return TQEnc(None,None,n,None,torch.zeros_like(x) if ret_deq else None,
                         float(n.numel()*4),'mse',0.0)
        y=u@self.W.T
        idx=torch.argmin(torch.abs(y.unsqueeze(-1)-self.centroids.view(1,1,-1)),dim=-1).to(torch.int16)
        xh=self.centroids[idx.long()]@self.W*n[:,None]
        return TQEnc(idx,None,n,None,xh if ret_deq else None,
                     float(idx.shape[0]*self.dim*self.bits/8+n.numel()*4),'mse',float(self.bits))
    @torch.no_grad()
    def dequantize(self, enc):
        if enc.indices is None:
            return torch.zeros(enc.norms.shape[0],self.dim,dtype=self.dtype)
        return self.centroids[enc.indices.long()]@self.W*enc.norms[:,None]

class TQProd(TQBase):
    """TurboQuant full (MSE + QJL 1-bit residual) — 논문 기준"""
    def __init__(self, dim, *, bits, seed=0, dtype=torch.float32):
        super().__init__(dim, bits=bits, seed=seed, dtype=dtype)
        self.mb=max(0,bits-1); self.mq=TQMse(dim,bits=self.mb,seed=seed,dtype=dtype)
        gen=torch.Generator(); gen.manual_seed(int(seed)+10000)
        self.S=torch.randn(dim,dim,generator=gen,dtype=torch.float32)
    @torch.no_grad()
    def encode(self, x, ret_deq=False):
        u,n=self._nr(x)
        if self.mb>0:
            me=self.mq.encode(u,ret_deq=True); xmh=me.dequantized; idx=me.indices
            mst=float(u.shape[0]*self.dim*self.mb/8)
        else:
            xmh=torch.zeros_like(u); idx=None; mst=0.0
        res=u-xmh
        rn=torch.clamp(torch.linalg.norm(res.float(),dim=-1,keepdim=True),min=1e-12).to(self.dtype)
        sgn=torch.sign((res.float()/rn.float())@self.S).to(torch.int8)
        rh=math.sqrt(math.pi/2)/self.dim*(sgn.float()@self.S)
        xh=(xmh+rh.to(self.dtype)*rn)*n[:,None]
        st=float(mst+u.shape[0]*self.dim/8+rn.numel()*4+n.numel()*4)
        return TQEnc(None if idx is None else idx,sgn,n,rn.squeeze(-1),
                     xh if ret_deq else None,st,'prod',float(self.bits))
    @torch.no_grad()
    def dequantize(self, enc):
        n=enc.norms
        xmh=(self.mq.dequantize(TQEnc(enc.indices,None,torch.ones_like(n),None,None,0.0,'mse',float(self.mb)))
             if self.mb>0 else torch.zeros(n.shape[0],self.dim,dtype=self.dtype))
        rh=math.sqrt(math.pi/2)/self.dim*(enc.qjl_signs.float()@self.S)
        return (xmh+rh.to(self.dtype)*enc.residual_norms[:,None])*n[:,None]

@dataclass
class SBlk:
    shape: Tuple; ke: TQEnc; ve: TQEnc

class TurboQuantKV:
    def __init__(self,*,B,H,D,bits=4,mode='mse',seed=7,dtype=torch.float32):
        self.B=B; self.H=H; self.D=D; self.dtype=dtype
        Q = TQMse if mode=='mse' else TQProd
        self.qk=Q(D,bits=bits,seed=seed+101,dtype=dtype)
        self.qv=Q(D,bits=bits,seed=seed+202,dtype=dtype)
        self.blocks: List[SBlk]=[]; self.total_bytes=0.0; self.mode=mode
    def clear(self): self.blocks.clear(); self.total_bytes=0.0
    @torch.no_grad()
    def update(self, nk, nv):
        B,H,T,D=nk.shape
        ke=self.qk.encode(nk.reshape(B*H*T,D).float())
        ve=self.qv.encode(nv.reshape(B*H*T,D).float())
        self.blocks.append(SBlk((B,H,T,D),ke,ve))
        self.total_bytes+=ke.storage_bytes+ve.storage_bytes
    @torch.no_grad()
    def materialize(self):
        if not self.blocks:
            e=torch.zeros(self.B,self.H,0,self.D,dtype=self.dtype); return e,e.clone()
        kl,vl=[],[]
        for b in self.blocks:
            Bb,Hb,Tb,Db=b.shape
            kl.append(self.qk.dequantize(b.ke).to(self.dtype).reshape(Bb,Hb,Tb,Db))
            vl.append(self.qv.dequantize(b.ve).to(self.dtype).reshape(Bb,Hb,Tb,Db))
        return torch.cat(kl,dim=2), torch.cat(vl,dim=2)
    def memory_bytes(self): return float(self.total_bytes)

class FullKVCache:
    def __init__(self,*,B,H,D,dtype=torch.float32):
        self.k=torch.zeros(B,H,0,D,dtype=dtype); self.v=torch.zeros_like(self.k)
    def update(self,nk,nv):
        self.k=torch.cat([self.k,nk.float()],dim=2)
        self.v=torch.cat([self.v,nv.float()],dim=2)
    def memory_bytes(self): return int((self.k.numel()+self.v.numel())*4)

@torch.no_grad()
def sdp_attn(q, k, v):
    if k.shape[2]==0: return torch.zeros_like(q)
    sc=1.0/math.sqrt(k.shape[-1])
    return torch.matmul(
        torch.softmax(torch.matmul(q.float(),k.float().transpose(-1,-2))*sc,dim=-1),
        v.float()).to(q.dtype)

print('TurboQuant (TQMse + TQProd) ✓')

TurboQuant (TQMse + TQProd) ✓


## Section 3 — NZFC Fix C

In [ ]:
@torch.no_grad()
def _proj_l1(v, r):
    w=v.float(); rt=torch.as_tensor(r,dtype=torch.float32)
    if torch.sum(w)<=rt: return w
    u,_=torch.sort(w,descending=True); cssv=torch.cumsum(u,dim=0)
    j=torch.arange(1,w.numel()+1,dtype=torch.float32)
    cond=u-(cssv-rt)/j>0
    if not torch.any(cond): return torch.zeros_like(v)
    rho=torch.nonzero(cond,as_tuple=False)[-1,0]
    return torch.clamp(w-(cssv[rho]-rt)/(rho.float()+1.0),min=0.0)

@torch.no_grad()
def _proj_nuc(A, tau):
    if min(A.shape)==0: return A.clone()
    U,s,Vh=torch.linalg.svd(A.float(),full_matrices=False)
    if torch.sum(s)<=tau: return A.float()
    return (U*_proj_l1(s,tau))@Vh

@torch.no_grad()
def _wkmeans(x, w, *, k, n_iters=6):
    n,d=x.shape
    if k>=n: return x.clone(), torch.arange(n,dtype=torch.long)
    xf=x.float(); wf=torch.clamp(w.float(),min=1e-8)
    C=torch.empty(k,d); C[0]=xf[torch.argmax(wf)]
    cl=torch.sum((xf-C[0])**2,dim=1)
    for j in range(1,k):
        p=wf*torch.clamp(cl,min=1e-12); tot=p.sum()
        idx=torch.multinomial(p/tot,1)[0] if (torch.isfinite(tot) and tot>0) else torch.argmax(cl)
        C[j]=xf[idx]; cl=torch.minimum(cl,torch.sum((xf-C[j])**2,dim=1))
    asgn=torch.zeros(n,dtype=torch.long)
    for _ in range(n_iters):
        D2=torch.cdist(xf,C,p=2.0)**2; na=torch.argmin(D2,dim=1)
        if torch.equal(na,asgn): break
        asgn=na
        for j in range(k):
            m=asgn==j
            if torch.any(m): C[j]=(wf[m][:,None]*xf[m]).sum(0)/wf[m].sum()
            else: C[j]=xf[torch.argmax(wf*torch.min(D2,dim=1).values)]
    return C, asgn

@torch.no_grad()
def _compress(rows, weights, *, tau, budget, sv_eps=1e-6, iters=6):
    m=weights>1e-6; rows=rows[m]; weights=weights[m]
    if rows.shape[0]==0: return rows.new_zeros((0,rows.shape[1])), weights.new_zeros((0,))
    n,fd=rows.shape; k=min(budget,n)
    if n<=budget:
        sw=torch.sqrt(weights.float())
        if torch.linalg.matrix_norm((sw[:,None]*rows).float(),ord='nuc')<=tau:
            o=torch.argsort(weights,descending=True); return rows[o],weights[o]
    sw=torch.sqrt(weights.float()); proj=_proj_nuc(sw[:,None]*rows,tau)
    U,s,_=torch.linalg.svd(proj.float(),full_matrices=False)
    keep=torch.nonzero(s>sv_eps,as_tuple=False).flatten()
    if keep.numel()==0:
        o=torch.argsort(weights,descending=True)[:k]; return rows[o],weights[o]
    _,asgn=_wkmeans(U[:,keep]*s[keep],weights,k=k,n_iters=iters)
    pts,ms=[],[]
    for ci in range(k):
        mm=asgn==ci
        if not torch.any(mm): continue
        ww=weights[mm][:,None]; pts.append((ww*rows[mm]).sum(0)/ww.sum())
        ms.append(weights[mm].sum())
    pt=torch.stack(pts,0) if pts else rows.new_zeros((0,fd))
    mt=torch.stack(ms,0) if ms else weights.new_zeros((0,))
    o=torch.argsort(mt,descending=True); return pt[o],mt[o]

def _mk_cache(B, H, budget, recent, D, dtype=torch.float32):
    ms = recent + 64
    return {
        'mk':torch.zeros(B,H,budget,D,dtype=dtype), 'mv':torch.zeros(B,H,budget,D,dtype=dtype),
        'mm':torch.zeros(B,H,budget,dtype=torch.float32),
        'ma':torch.zeros(B,H,dtype=torch.long),
        'rk':torch.zeros(B,H,recent,D,dtype=dtype),  'rv':torch.zeros(B,H,recent,D,dtype=dtype),
        'ra':torch.zeros(B,H,dtype=torch.long),
        'ts':torch.zeros(B,H,dtype=torch.long),
        'B':B,'H':H,'D':D,'budget':budget,'recent':recent,'dtype':dtype,
        '_buf_cnt': torch.zeros(budget),
        '_buf_sk':  torch.zeros(budget,D), '_buf_sv':  torch.zeros(budget,D),
        '_buf_ones':torch.ones(ms),
        '_buf_ae':  torch.zeros(ms,D,dtype=torch.long),
    }

@torch.no_grad()
def _recompress(c, bi, hi, sk, sv, tau, decay=0.99, boost=1.05):
    D=c['D']; on=int(c['ma'][bi,hi].item())
    or_=(torch.cat([c['mk'][bi,hi,:on],c['mv'][bi,hi,:on]],dim=-1) if on>0
         else sk.new_zeros((0,2*D)))
    ow=c['mm'][bi,hi,:on]*decay if on>0 else torch.zeros((0,))
    sn=sk.shape[0]
    sr=torch.cat([sk,sv],dim=-1) if sn>0 else sk.new_zeros((0,2*D))
    sw=torch.full((sn,),boost) if sn>0 else torch.zeros((0,))
    rows=torch.cat([or_,sr],dim=0); weights=torch.cat([ow,sw],dim=0)
    if rows.shape[0]==0:
        c['mk'][bi,hi].zero_(); c['mv'][bi,hi].zero_()
        c['mm'][bi,hi].zero_(); c['ma'][bi,hi]=0; return
    p,m=_compress(rows,weights,tau=tau,budget=c['budget'])
    ac=p.shape[0]
    c['mk'][bi,hi].zero_(); c['mv'][bi,hi].zero_(); c['mm'][bi,hi].zero_()
    if ac>0:
        c['mk'][bi,hi,:ac]=p[:,:D].to(c['dtype'])
        c['mv'][bi,hi,:ac]=p[:,D:].to(c['dtype'])
        c['mm'][bi,hi,:ac]=m.float()
    c['ma'][bi,hi]=ac

@torch.no_grad()
def _online_fixc(c, bi, hi, sk, sv, decay=0.99, boost=1.05):
    if sk.shape[0]==0: return
    D=c['D']; np2=int(c['ma'][bi,hi].item())
    # Phase 1: budget 미달 → 직접 추가
    if np2 < c['budget']:
        slots=min(c['budget']-np2, sk.shape[0])
        c['mk'][bi,hi,np2:np2+slots]=sk[:slots].float()
        c['mv'][bi,hi,np2:np2+slots]=sv[:slots].float()
        c['mm'][bi,hi,np2:np2+slots]=boost
        c['ma'][bi,hi]=np2+slots
        if np2>0 and decay!=1.0: c['mm'][bi,hi,:np2]*=decay
        sk=sk[slots:]; sv=sv[slots:]; np2=int(c['ma'][bi,hi].item())
        if sk.shape[0]==0: return
    # Phase 2: 완전 벡터화 (scatter_add_ + K-only cdist + pre-alloc)
    ns=sk.shape[0]
    pk=c['mk'][bi,hi,:np2]; pv=c['mv'][bi,hi,:np2]; pm=c['mm'][bi,hi,:np2]
    asgn=torch.argmin(torch.cdist(sk.float(),pk.float(),p=2.0),dim=-1)
    cnt=c['_buf_cnt'][:np2]; cnt.zero_()
    cnt.scatter_add_(0,asgn,c['_buf_ones'][:ns])
    valid=cnt>0
    if not valid.any(): pm.mul_(decay); return
    ae=c['_buf_ae'][:ns]; ae.copy_(asgn.unsqueeze(1).expand(-1,D))
    sk_s=c['_buf_sk'][:np2]; sk_s.zero_(); sk_s.scatter_add_(0,ae,sk.float())
    sv_s=c['_buf_sv'][:np2]; sv_s.zero_(); sv_s.scatter_add_(0,ae,sv.float())
    inv_c=cnt.clamp(min=1).reciprocal_()
    wi_m=cnt.mul(boost); new_pm=pm.mul(decay).add_(wi_m)
    a=torch.where(valid,wi_m.div(new_pm.clamp(min=1e-9)),torch.zeros(np2))
    a1=a.unsqueeze(1); inv1=inv_c.unsqueeze(1)
    pk.mul_(1-a1).add_(a1.mul(sk_s).mul_(inv1))
    pv.mul_(1-a1).add_(a1.mul(sv_s).mul_(inv1))
    pm.copy_(torch.where(valid,new_pm,pm.mul(decay)))

@torch.no_grad()
def nzfc_update(c, nk, nv, tau, variant='fixc', decay=0.99, boost=1.05):
    B,H,T,D=nk.shape
    for bi in range(B):
        for hi in range(H):
            cur=int(c['ra'][bi,hi].item())
            ck=torch.cat([c['rk'][bi,hi,:cur],nk[bi,hi].float()],dim=0)
            cv=torch.cat([c['rv'][bi,hi,:cur],nv[bi,hi].float()],dim=0)
            kr=min(c['recent'],ck.shape[0]); sp=ck.shape[0]-kr
            spk=ck[:sp] if sp>0 else ck.new_zeros((0,D))
            spv=cv[:sp] if sp>0 else cv.new_zeros((0,D))
            c['rk'][bi,hi].zero_(); c['rv'][bi,hi].zero_()
            if kr>0: c['rk'][bi,hi,:kr]=ck[sp:]; c['rv'][bi,hi,:kr]=cv[sp:]
            c['ra'][bi,hi]=kr
            if variant=='orig':
                if sp>0: _recompress(c,bi,hi,spk,spv,tau,decay,boost)
            else:
                _online_fixc(c,bi,hi,spk,spv,decay,boost)
            c['ts'][bi,hi]+=T

@torch.no_grad()
def nzfc_attend(c, query, alpha=1.0):
    q=query.float(); B,H,_,D=q.shape; out=torch.zeros_like(q); sc=1.0/math.sqrt(D)
    for bi in range(B):
        for hi in range(H):
            sp,vp=[],[]
            mn=int(c['ma'][bi,hi].item())
            if mn>0:
                km=c['mk'][bi,hi,:mn].float(); vm=c['mv'][bi,hi,:mn].float()
                mass=torch.clamp(c['mm'][bi,hi,:mn].float(),min=1e-6)
                sm=(q[bi,hi]@km.T)*sc
                if alpha>0: sm+=alpha*torch.log(mass)[None,:]
                sp.append(sm); vp.append(vm)
            rn=int(c['ra'][bi,hi].item())
            if rn>0:
                sp.append((q[bi,hi]@c['rk'][bi,hi,:rn].float().T)*sc)
                vp.append(c['rv'][bi,hi,:rn].float())
            if not sp: continue
            out[bi,hi]=(torch.softmax(torch.cat(sp,dim=-1),dim=-1)@torch.cat(vp,dim=0))
    return out

def rel_l2(x, y):
    return float((torch.linalg.norm((x-y).float().reshape(-1)) /
                  torch.clamp(torch.linalg.norm(y.float().reshape(-1)),min=1e-12)).item())

def cos_sim(x, y):
    xf=x.float().reshape(-1); yf=y.float().reshape(-1)
    return float((torch.dot(xf,yf)/torch.clamp(xf.norm()*yf.norm(),min=1e-12)).item())

def estimate_tau(ks, vs, calib=256, q=0.8):
    kl,vl,tok=[],[],0
    for k,v in zip(ks,vs):
        kl.append(k.float()); vl.append(v.float()); tok+=k.shape[2]
        if tok>=calib: break
    K=torch.cat(kl,dim=2); V=torch.cat(vl,dim=2)
    nuc=[float(torch.linalg.matrix_norm(torch.cat([K[b,h],V[b,h]],dim=-1),ord='nuc').item())
         for b in range(K.shape[0]) for h in range(K.shape[1])]
    return max(float(np.quantile(np.array(nuc),q)),1e-6)

print('NZFC Fix C ✓')

NZFC Fix C ✓


## Section 4 — Sweep

In [ ]:
@torch.no_grad()
def run_layer(ks, vs, *, ctx, tau, budget, recent, B, H, D, bits=4, seed=7):
    cks,cvs=[],[]; seen=0
    for k,v in zip(ks,vs):
        take=min(k.shape[2], ctx-seen)
        if take<=0: break
        cks.append(k[:,:,:take]); cvs.append(v[:,:,:take]); seen+=take
        if seen>=ctx: break
    if not cks: return None

    t_mse  = TurboQuantKV(B=B,H=H,D=D,bits=bits,mode='mse', seed=seed)
    t_prod = TurboQuantKV(B=B,H=H,D=D,bits=bits,mode='prod',seed=seed)
    full   = FullKVCache(B=B,H=H,D=D)
    c_orig = _mk_cache(B,H,budget,recent,D)
    c_fixc = _mk_cache(B,H,budget,recent,D)

    m={k:[] for k in ['orig_l2','fixc_l2','tmse_l2','tprod_l2',
                       'orig_cos','fixc_cos','tmse_cos','tprod_cos',
                       'orig_upd','fixc_upd','tmse_upd','tprod_upd',
                       'orig_qry','fixc_qry','tmse_qry','tprod_qry']}

    for k,v in zip(cks,cvs):
        k=k.float(); v=v.float()
        full.update(k,v)
        t0=time.perf_counter(); t_mse.update(k,v);                          m['tmse_upd'].append((time.perf_counter()-t0)*1e3)
        t0=time.perf_counter(); t_prod.update(k,v);                         m['tprod_upd'].append((time.perf_counter()-t0)*1e3)
        t0=time.perf_counter(); nzfc_update(c_orig,k,v,tau,variant='orig'); m['orig_upd'].append((time.perf_counter()-t0)*1e3)
        t0=time.perf_counter(); nzfc_update(c_fixc,k,v,tau,variant='fixc'); m['fixc_upd'].append((time.perf_counter()-t0)*1e3)

        T_cur=full.k.shape[2]; qn=min(8,T_cur)
        qp=full.k[:,:,T_cur-qn:]; ref=sdp_attn(qp,full.k,full.v)

        t0=time.perf_counter(); mk,mv=t_mse.materialize();  tmout=sdp_attn(qp,mk,mv);  m['tmse_qry'].append((time.perf_counter()-t0)*1e3)
        t0=time.perf_counter(); pk,pv=t_prod.materialize(); tpout=sdp_attn(qp,pk,pv);  m['tprod_qry'].append((time.perf_counter()-t0)*1e3)
        t0=time.perf_counter(); no=nzfc_attend(c_orig,qp);                              m['orig_qry'].append((time.perf_counter()-t0)*1e3)
        t0=time.perf_counter(); nc=nzfc_attend(c_fixc,qp);                              m['fixc_qry'].append((time.perf_counter()-t0)*1e3)

        for tag,out in [('tmse',tmout),('tprod',tpout),('orig',no),('fixc',nc)]:
            m[f'{tag}_l2'].append(rel_l2(out,ref))
            m[f'{tag}_cos'].append(cos_sim(out,ref))

    fm=full.memory_bytes()
    e2e={t: float(np.mean(m[f'{t}_upd']))+float(np.mean(m[f'{t}_qry']))
         for t in ['tmse','tprod','orig','fixc']}
    return {
        'ctx':ctx,'budget':budget,'recent':recent,'tau':tau,
        'full_bytes':fm,
        'tmse_bytes':float(t_mse.total_bytes), 'tprod_bytes':float(t_prod.total_bytes),
        'tmse_compr':fm/max(float(t_mse.total_bytes),1),
        **{k:float(np.mean(v)) for k,v in m.items()},
        'e2e_tmse':e2e['tmse'],'e2e_tprod':e2e['tprod'],
        'e2e_orig':e2e['orig'],'e2e_fixc':e2e['fixc'],
    }

CONTEXTS   = [256, 512, 1024]
BUDGETS    = [24, 48, 96]
RECENTS    = [16, 32]
TAU_SCALES = [0.75, 1.0, 1.5, 2.0]

all_rows=[]; n=0
total=len(BENCH_LAYERS)*len(CONTEXTS)*len(BUDGETS)*len(RECENTS)*len(TAU_SCALES)
print(f'Starting sweep ({total} runs)...')

for li in BENCH_LAYERS:
    ks=kv_dump[li]['k_chunks']; vs=kv_dump[li]['v_chunks']
    if not ks: print(f'  SKIP layer {li}'); continue
    base_tau=estimate_tau(ks,vs)
    for ctx in CONTEXTS:
        avail=sum(k.shape[2] for k in ks)
        if ctx>avail: continue
        for budget in BUDGETS:
            for recent in RECENTS:
                for scale in TAU_SCALES:
                    n+=1
                    try:
                        r=run_layer(ks,vs,ctx=ctx,tau=float(scale*base_tau),
                                    budget=budget,recent=recent,
                                    B=1,H=N_KV_HEADS,D=HEAD_DIM)
                        if r:
                            r.update({'layer':li,'tau_scale':float(scale),'source':'real_qwen2.5-7B'})
                            all_rows.append(r)
                    except Exception as e:
                        print(f'[ERR] L{li} ctx={ctx}: {e}')
                    if n%48==0:
                        print(f'[{n}/{total}] L{li} ctx={ctx} budget={budget} tau*{scale:.2f}')

df=pd.DataFrame(all_rows)
df.to_csv(OUT_DIR/'sweep_qwen25_7b.csv',index=False)
print(f'\n✓ {len(df)} rows saved')
if len(df)==0:
    print('WARNING: df empty — kv_dump 상태 확인:')
    for li in BENCH_LAYERS:
        chunks=kv_dump[li]['k_chunks']
        print(f'  L{li}: {len(chunks)} chunks, {sum(c.shape[2] for c in chunks) if chunks else 0} tokens')

Starting sweep (288 runs)...
[48/288] L6 ctx=512 budget=96 tau*2.00
[96/288] L13 ctx=256 budget=96 tau*2.00


## Section 5 — 결과 테이블

In [ ]:
if len(df)==0:
    print('ERROR: sweep 결과 없음 — sweep 셀 출력을 확인하세요')
else:
    summary_rows=[]
    for ctx,g in df.groupby('ctx'):
        summary_rows.append({
            'ctx':int(ctx),
            'tmse_l2':   round(float(g['tmse_l2'].mean()),4),
            'tprod_l2':  round(float(g['tprod_l2'].mean()),4),
            'orig_l2':   round(float(g['orig_l2'].min()),4),
            'fixc_l2':   round(float(g['fixc_l2'].min()),4),
            'tmse_upd':  round(float(g['tmse_upd'].mean()),2),
            'tprod_upd': round(float(g['tprod_upd'].mean()),2),
            'orig_upd':  round(float(g['orig_upd'].mean()),2),
            'fixc_upd':  round(float(g['fixc_upd'].mean()),2),
            'tmse_qry':  round(float(g['tmse_qry'].mean()),2),
            'fixc_qry':  round(float(g['fixc_qry'].mean()),2),
            'e2e_tmse':  round(float(g['e2e_tmse'].mean()),2),
            'e2e_tprod': round(float(g['e2e_tprod'].mean()),2),
            'e2e_fixc':  round(float(g['e2e_fixc'].mean()),2),
            'tmse_compr':round(float(g['tmse_compr'].mean()),2),
        })
    summary=pd.DataFrame(summary_rows)
    summary.to_csv(OUT_DIR/'summary_qwen25_7b.csv',index=False)

    print('=== 품질 (rel-L2) ===')
    display(summary[['ctx','tmse_l2','tprod_l2','orig_l2','fixc_l2']].round(4))

    print('\n=== Update latency (ms/step) ===')
    lat=summary[['ctx','tmse_upd','tprod_upd','orig_upd','fixc_upd']].copy()
    lat['fixc/tmse']=(lat['fixc_upd']/lat['tmse_upd']).round(2)
    display(lat)

    print('\n=== Query latency (ms/step) ===')
    qry=summary[['ctx','tmse_qry','fixc_qry']].copy()
    qry['tmse/fixc']=(qry['tmse_qry']/qry['fixc_qry']).round(1)
    display(qry)

    print('\n=== End-to-end (ms/step) ===')
    e2e=summary[['ctx','e2e_tmse','e2e_tprod','e2e_fixc']].copy()
    e2e['fixc/tmse']=(e2e['e2e_tmse']/e2e['e2e_fixc']).round(1)
    e2e['fixc_wins']=e2e['e2e_fixc']<e2e['e2e_tmse']
    display(e2e)

## Section 6 — 시각화

In [ ]:
if len(df)>0:
    CTX_LIST=[256,512,1024]
    COLS={'tmse':'#3B8BD4','tprod':'#82B8E8','orig':'#E24B4A','fixc':'#1D9E75'}
    LABS={'tmse':'TQ-MSE (강한 기준)','tprod':'TQ-Prod (논문)','orig':'NZFC-Orig','fixc':'NZFC-FixC'}

    fig,axes=plt.subplots(1,3,figsize=(16,5))
    fig.suptitle('NZFC Fix C vs TurboQuant — Qwen2.5-7B  |  H_KV=8  head_dim=128',
                 fontsize=12,fontweight='bold')

    for ax,metric,title,best_fn in [
        (axes[0],'upd','Update latency (ms/step)',np.mean),
        (axes[1],'l2', 'Quality rel-L2 (lower=better)',np.mean),
        (axes[2],'e2e','End-to-end (ms/step)',np.mean),
    ]:
        for var,ls in [('orig','-'),('fixc','-'),('tmse','--'),('tprod','--')]:
            if metric=='e2e':
                col=f'e2e_{var}'
                vals=[df[df.ctx==c][col].mean() for c in CTX_LIST]
            elif metric=='upd':
                col=f'{var}_upd'
                vals=[df[df.ctx==c][col].mean() for c in CTX_LIST]
            else:
                col=f'{var}_l2'
                vals=[df[df.ctx==c][col].min() if var in ['orig','fixc']
                      else df[df.ctx==c][col].mean() for c in CTX_LIST]
            ax.plot(CTX_LIST,vals,'o',linestyle=ls,color=COLS[var],label=LABS[var],linewidth=2)
        ax.set_xlabel('Context tokens'); ax.set_title(title)
        ax.legend(fontsize=8); ax.grid(True,alpha=0.25)

    fig.tight_layout()
    fig.savefig(OUT_DIR/'comparison_qwen25_7b.png',dpi=150,bbox_inches='tight')
    plt.show()

    # Layer-wise quality
    fig2,axes2=plt.subplots(1,len(BENCH_LAYERS),figsize=(14,4),sharey=True)
    fig2.suptitle('Layer별 품질 (ctx=512, budget=48)', fontsize=11)
    for ax,li in zip(axes2,BENCH_LAYERS):
        sub=df[(df.layer==li)&(df.ctx==512)&(df.budget==48)]
        if sub.empty: continue
        labels=['TQ-MSE','TQ-Prod','Orig','FixC']
        vals=[sub['tmse_l2'].mean(),sub['tprod_l2'].mean(),sub['orig_l2'].min(),sub['fixc_l2'].min()]
        colors=[COLS['tmse'],COLS['tprod'],COLS['orig'],COLS['fixc']]
        ax.bar(labels,vals,color=colors,alpha=0.8)
        ax.set_title(f'Layer {li}'); ax.set_ylabel('rel-L2'); ax.tick_params(axis='x',rotation=30)
    fig2.tight_layout()
    fig2.savefig(OUT_DIR/'layerwise_qwen25_7b.png',dpi=150,bbox_inches='tight')
    plt.show()

## Section 7 — cProfile

In [ ]:
import cProfile, pstats, io

li_prof=BENCH_LAYERS[1]
ks_p=kv_dump[li_prof]['k_chunks']; vs_p=kv_dump[li_prof]['v_chunks']
if not ks_p:
    print('Profiling 스킵 — KV dump 없음')
else:
    tau_p=estimate_tau(ks_p,vs_p)
    for label,variant in [('NZFC-FixC','fixc'),('NZFC-Orig','orig')]:
        for ctx in [512,1024]:
            c=_mk_cache(1,N_KV_HEADS,48,32,HEAD_DIM)
            cks2,cvs2=[],[]; seen=0
            for k,v in zip(ks_p,vs_p):
                take=min(k.shape[2],ctx-seen)
                if take<=0: break
                cks2.append(k[:,:,:take]); cvs2.append(v[:,:,:take]); seen+=take
                if seen>=ctx: break
            if not cks2: continue
            pr=cProfile.Profile(); pr.enable()
            for k,v in zip(cks2,cvs2):
                nzfc_update(c,k.float(),v.float(),tau_p,variant=variant)
            pr.disable()
            s=io.StringIO(); pstats.Stats(pr,stream=s).sort_stats('tottime').print_stats(8)
            print(f'\n── {label}  ctx={ctx}  H={N_KV_HEADS}  D={HEAD_DIM}  budget=48 ──')
            for l in s.getvalue().split('\n'):
                if any(x in l for x in ['tottime','ncalls','cdist','scatter','fixc','recompress','cumtime']):
                    print(l)

## Section 8 — 최종 리포트

In [ ]:
if len(df)==0:
    print('ERROR: 결과 없음')
else:
    lines=[f'# LUMENSI Stage-2 Fix C — Qwen2.5-7B', '']
    lines.append(f'- model: `Qwen/Qwen2.5-7B`  kv_heads={N_KV_HEADS}  head_dim={HEAD_DIM}')
    lines.append(f'- layers: {BENCH_LAYERS}  tokens: {MAX_TOKENS}  source: wikitext-103')
    lines.append('')
    lines.append('## 비교 기준')
    lines.append('| 방법 | 설명 |')
    lines.append('|---|---|')
    lines.append('| TQ-MSE  | TurboQuant MSE-only (QJL 없음) — 공정한 강한 기준선 |')
    lines.append('| TQ-Prod | 논문 TurboQuant (MSE + 1-bit QJL) |')
    lines.append('| NZFC-FixC | scatter_add_ + K-only cdist + pre-alloc |')
    lines.append('')
    lines.append('## 실측 결과')
    lines.append(summary[['ctx','tmse_l2','tprod_l2','fixc_l2',
                           'tmse_upd','fixc_upd','tmse_qry','fixc_qry',
                           'e2e_tmse','e2e_fixc']].to_markdown(index=False,floatfmt='.3f'))
    lines.append('')
    lines.append('## 컨텍스트별 판정 (TQ-MSE 기준)')
    for _,row in summary.iterrows():
        ctx=int(row['ctx'])
        q_win  = '✅ FixC 우수' if row['fixc_l2'] < row['tmse_l2']  else '❌ TQ-MSE 우수'
        e_win  = '✅ FixC 우수' if row['e2e_fixc'] < row['e2e_tmse'] else '❌ TQ-MSE 우수'
        spdup  = row['e2e_tmse']/row['e2e_fixc']
        lines.append(f'- ctx={ctx}: 품질={q_win}  e2e={e_win}  speedup={spdup:.1f}×')

    rpt=OUT_DIR/'LUMENSI_7B_REPORT.md'
    rpt.write_text('\n'.join(lines),encoding='utf-8')
    display(Markdown('\n'.join(lines)))

    print('\n✓ artifacts:', [f.name for f in sorted(OUT_DIR.iterdir())])